# 03. Market Screener Signal Backtesting
Test multi-factor screener rules (RSI Oversold Bounce, Bullish MACD Crossover, Volume Surge) across historical klines to evaluate win rate and average gain.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe

## 1. Run Live Screener Engine

In [ ]:
from backend.screener import screener_engine

screener_results = screener_engine.run_screener(interval='1h', max_pairs=30)
df_screener = pd.DataFrame(screener_results)
df_screener[['symbol', 'category', 'price', 'rsi', 'rvol', 'score', 'recommendation', 'signals']].head(15)

## 2. Historical Signal Performance Simulation

In [ ]:
# Fetch 500 candles for BTCUSDT to test Bullish MACD Crossover signal
raw_klines = binance_client.get_klines('BTCUSDT', interval='1h', limit=500)
df_btc = enrich_klines_dataframe(raw_klines)

# Identify MACD Bullish Crossovers (Prev Hist <= 0 and Curr Hist > 0)
df_btc['prev_macd_hist'] = df_btc['macd_hist'].shift(1)
df_btc['bull_cross'] = (df_btc['prev_macd_hist'] <= 0) & (df_btc['macd_hist'] > 0)

# Forward return over 6 hours
df_btc['future_close_6h'] = df_btc['close'].shift(-6)
df_btc['return_6h_%'] = ((df_btc['future_close_6h'] - df_btc['close']) / df_btc['close']) * 100

signals_df = df_btc[df_btc['bull_cross'] == True].dropna(subset=['return_6h_%'])
win_rate = (signals_df['return_6h_%'] > 0).mean() * 100
avg_return = signals_df['return_6h_%'].mean()

print(f"Total Bullish MACD Cross Signals Triggered: {len(signals_df)}")
print(f"Win Rate (Positive 6h Return): {win_rate:.2f}%")
print(f"Average 6h Return per Signal: {avg_return:.2f}%")